# Solution: Automated EDA — Swiss EV Charging Stations

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Task 1
df = pd.read_csv('../data/ev_charging_stations.csv')
print('Shape:', df.shape)
df.head(3)

### Task 2 — `sweetviz` report

In [ ]:
import logging
# Silence 'findfont: Failed to find font ... Roboto' messages emitted while sweetviz renders
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)

try:
    import sweetviz as sv
    report = sv.analyze(df)
    report.show_html('ev_sweetviz_report.html', open_browser=False)
    print('Report saved to ev_sweetviz_report.html')
except ImportError:
    print('sweetviz not installed. Run: pip install sweetviz')

### Task 3 — Lightweight `quick_profile()`

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def quick_profile(df: pd.DataFrame, max_cols: int = 10) -> None:
    num_cols = df.select_dtypes(include='number').columns[:max_cols]
    print('=== Quick Profile ===')
    print(f'Rows: {len(df):,}  |  Columns: {df.shape[1]}')
    print(f'Missing values:\n{df[num_cols].isnull().sum().to_string()}')
    print(f'\nDuplicates: {df.duplicated().sum()}')
    print('\nDescriptive statistics:')
    display(df[num_cols].describe().round(2))
    n = len(num_cols)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 3))
    if n == 1:
        axes = [axes]
    for ax, col in zip(axes, num_cols):
        sns.histplot(df[col].dropna(), bins=30, kde=True, ax=ax)
        ax.set_title(col)
        ax.set_xlabel('')
    fig.suptitle('Variable Distributions', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.show()

quick_profile(df)

### Task 4 — Answer the questions

In [ ]:
# Which variable has the highest skewness?
num_cols = ['power_kw', 'amperage', 'voltage']
skew = df[num_cols].skew().sort_values(ascending=False)
print('Skewness:')
print(skew.round(3))
print(f'\nHighest skewness: {skew.index[0]} ({skew.iloc[0]:.2f})')
print('power_kw is strongly right-skewed: most stations provide 22 kW, but a few fast-chargers reach 400 kW.')

In [ ]:
# Are there any missing values or duplicates?
print('Missing values per column:')
print(df.isnull().sum())
print(f'\nDuplicate rows: {df.duplicated().sum()}')
print('\nNo missing values and no duplicates in this dataset.')

In [ ]:
# Which two variables are most strongly correlated?
corr = df[num_cols].corr(method='pearson')
print('Pearson correlation matrix:')
print(corr.round(3))

# Extract off-diagonal maximum
import numpy as np
mask = np.triu(np.ones(corr.shape), k=1).astype(bool)
upper = corr.where(mask)
max_pair = upper.stack().abs().idxmax()
max_val = upper.loc[max_pair]
print(f'\nStrongest correlation: {max_pair[0]} vs. {max_pair[1]} (r = {max_val:.3f})')
print('Higher charging power requires higher current (P = U x I).')

## Jupyter notebook --footer info--
(please always provide this at the end of each notebook)

In [ ]:
import os, platform, socket
from platform import python_version
from datetime import datetime
print('------------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('------------------------------------')